# Task 1 - HAB prediction: train/test split, median imputation, log transform and z-score normalization

This notebook prepares the data for modelling in four steps, in this order:

1. Split the data 80/20 into train and test sets.
2. Fill the missing `CHL` and `CHL_uncertainty` values with medians computed on the train set only.
3. Apply a log10 transform to `CHL`, which has a long right tail.
4. Z-score normalize the continuous features with means and standard deviations computed on the train set only.

Computing these values after the split, from the train set alone, stops information from the test set leaking into training.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("classification_task1_train.csv")
TARGET = "HAB"
IMPUTE_COLUMNS = ["CHL", "CHL_uncertainty"]
LOG_COLUMN = "CHL"
SCALE_COLUMNS = ["CHL", "CHL_uncertainty", "SST", "UI", "YDOY"]
TRAIN_SHARE = 0.8

## 1. Load the data

In [2]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", parse_dates=["Date"])
df = df.sort_values("Date", kind="stable").reset_index(drop=True)

print(f"{df.shape[0]} rows, {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d}")
df.isna().sum().to_frame("missing values")

1063 rows, 2013-02-19 to 2020-12-29


,missing values
Date,0
Day,0
Month,0
Year,0
Season,0
CHL,500
CHL_uncertainty,500
SST,0
UI,0
YDOY,0


## 2. Chronological 80/20 split

The split is chronological because the manual asks for a chronological validation split and the model will be evaluated on
later years (2021-2023). The earliest rows form the train set and the latest rows form the test set.

Some dates have several rows, one for each spatial cluster sampled that day. The cut is placed on a date boundary so that a
date never appears in both sets, which makes the split close to 80/20 rather than exact.

In [3]:
cutoff_date = df["Date"].iloc[int(len(df) * TRAIN_SHARE)]
train = df[df["Date"] < cutoff_date]
test = df[df["Date"] >= cutoff_date]


def split_summary(data):
    """Return the size, date range, HAB rate and missing CHL share of a split."""
    return {
        "rows": len(data),
        "share (%)": round(len(data) / len(df) * 100, 1),
        "first date": f"{data['Date'].min():%Y-%m-%d}",
        "last date": f"{data['Date'].max():%Y-%m-%d}",
        "positives": data[TARGET].sum(),
        "HAB rate (%)": round(data[TARGET].mean() * 100, 1),
        "missing CHL (%)": round(data["CHL"].isna().mean() * 100, 1),
    }


pd.DataFrame({"Train": split_summary(train), "Test": split_summary(test)}).T

,rows,share (%),first date,last date,positives,HAB rate (%),missing CHL (%)
Train,850,80.0,2013-02-19,2019-08-19,98,11.5,46.8
Test,213,20.0,2019-08-21,2020-12-29,40,18.8,47.9


## 3. Median imputation

The medians come from the train set only, and the same values fill the gaps in both sets. This uses plain pandas because the
manual doesn't allow library imputers in the submission.

In [4]:
train_medians = train[IMPUTE_COLUMNS].median()

train_imputed = train.fillna(train_medians)
test_imputed = test.fillna(train_medians)

pd.DataFrame({
    "train median (used to fill)": train_medians,
    "test median (not used)": test[IMPUTE_COLUMNS].median(),
    "filled in train": train[IMPUTE_COLUMNS].isna().sum(),
    "filled in test": test[IMPUTE_COLUMNS].isna().sum(),
}).round(2)

,train median (used to fill),test median (not used),filled in train,filled in test
CHL,2.89,2.98,398,102
CHL_uncertainty,32.45,34.25,398,102


## 4. Check the result

Each check stops the notebook with an error if it fails.

In [5]:
for name, original, imputed in [("Train", train, train_imputed), ("Test", test, test_imputed)]:
    assert imputed.notna().all().all(), f"{name}: missing values remain"
    assert len(imputed) == len(original), f"{name}: row count changed"
    for column in IMPUTE_COLUMNS:
        was_missing = original[column].isna()
        assert (imputed.loc[was_missing, column] == train_medians[column]).all(), f"{name}: wrong fill for {column}"
        assert imputed.loc[~was_missing, column].equals(original.loc[~was_missing, column]), f"{name}: {column} changed"

assert len(train) + len(test) == len(df), "Rows were lost in the split"
assert train["Date"].max() < test["Date"].min(), "Train and test overlap in time"

print("All checks passed")

All checks passed


## 5. Log transform of CHL

`CHL` becomes `log10(CHL)` in both sets, after imputation and before scaling, so filled rows get the log of the train median.
The transform has no fitted values, so it can't leak information between the sets. It needs every value to be positive,
which the first check below confirms. `CHL_uncertainty` isn't transformed.

In [6]:
def log_chl(data):
    """Return a copy of the data with LOG_COLUMN replaced by its log10."""
    assert (data[LOG_COLUMN] > 0).all(), "log10 needs positive values"
    logged = data.copy()
    logged[LOG_COLUMN] = np.log10(data[LOG_COLUMN])
    return logged


train_logged = log_chl(train_imputed)
test_logged = log_chl(test_imputed)

for imputed, logged in [(train_imputed, train_logged), (test_imputed, test_logged)]:
    assert ((10 ** logged[LOG_COLUMN] - imputed[LOG_COLUMN]).abs() < 1e-9).all(), "log10 can't be reversed"
    assert logged.drop(columns=LOG_COLUMN).equals(imputed.drop(columns=LOG_COLUMN)), "Another column changed"

pd.DataFrame({
    "skewness before log": [train_imputed[LOG_COLUMN].skew(), test_imputed[LOG_COLUMN].skew()],
    "skewness after log": [train_logged[LOG_COLUMN].skew(), test_logged[LOG_COLUMN].skew()],
}, index=["Train", "Test"]).round(2)

,skewness before log,skewness after log
Train,5.40,0.98
Test,5.53,0.87


## 6. Z-score normalization

Each continuous feature becomes `(value - train mean) / train std`, using the log-transformed train set for the mean and
standard deviation, so `CHL` is scaled on its log10 values. The test set is scaled with the same train values, so its means and standard deviations won't be exactly 0 and 1.

These columns are left as they are:
- `HAB` is the target.
- `Season` and `spatial_cluster` are categories, so their numeric distance has no meaning.
- `Date`, `Day`, `Month` and `Year` are calendar columns, which the manual describes as redundant.

The means and standard deviations are stored as plain values, which the manual allows in `model.pkl`, so prediction can reuse them.

In [7]:
train_means = train_logged[SCALE_COLUMNS].mean()
train_stds = train_logged[SCALE_COLUMNS].std(ddof=0)
assert (train_stds > 0).all(), "A feature is constant in the train set"


def z_score(data):
    """Return a copy of the data with SCALE_COLUMNS scaled by the train statistics."""
    scaled = data.copy()
    scaled[SCALE_COLUMNS] = (data[SCALE_COLUMNS] - train_means) / train_stds
    return scaled


train_scaled = z_score(train_logged)
test_scaled = z_score(test_logged)

pd.DataFrame({
    "train mean (used)": train_means,
    "train std (used)": train_stds,
    "scaled train mean": train_scaled[SCALE_COLUMNS].mean(),
    "scaled train std": train_scaled[SCALE_COLUMNS].std(ddof=0),
    "scaled test mean": test_scaled[SCALE_COLUMNS].mean(),
    "scaled test std": test_scaled[SCALE_COLUMNS].std(ddof=0),
}).round(3)

,train mean (used),train std (used),scaled train mean,scaled train std,scaled test mean,scaled test std
CHL,0.501,0.303,0.0,1.0,-0.089,0.993
CHL_uncertainty,35.816,15.899,0.0,1.0,0.037,1.071
SST,14.657,1.717,-0.0,1.0,-0.030,0.883
UI,0.064,0.381,-0.0,1.0,-0.125,0.940
YDOY,-0.060,0.693,0.0,1.0,0.177,0.979


## 7. Check the normalization

Each check stops the notebook with an error if it fails.

In [8]:
other_columns = [column for column in df.columns if column not in SCALE_COLUMNS]

assert (train_scaled[SCALE_COLUMNS].mean().abs() < 1e-9).all(), "Scaled train means are not 0"
assert ((train_scaled[SCALE_COLUMNS].std(ddof=0) - 1).abs() < 1e-9).all(), "Scaled train stds are not 1"

for name, logged, scaled in [("Train", train_logged, train_scaled), ("Test", test_logged, test_scaled)]:
    assert scaled[SCALE_COLUMNS].notna().all().all(), f"{name}: scaling created missing values"
    assert scaled[other_columns].equals(logged[other_columns]), f"{name}: an unscaled column changed"
    restored = scaled[SCALE_COLUMNS] * train_stds + train_means
    assert ((restored - logged[SCALE_COLUMNS]).abs() < 1e-9).all().all(), f"{name}: scaling can't be reversed"

print("All normalization checks passed")

All normalization checks passed


## 8. Notes for modelling

`train_scaled` and `test_scaled` are ready for modelling.

- **Class balance:** the test set has more positives than the train set (18.8% against 11.5% HAB rate), so scores may not carry over directly between periods.
- **Effect of imputation:** about 47% of rows in each set get the same CHL value, which adds a spike at the median and shrinks the spread of CHL.
- **Matching the real evaluation:** the manual's test set has only complete samples, but 102 of the 213 test rows here are imputed. Scoring also on the 111 complete rows, `test_scaled[test["CHL"].notna()]`, gives a closer match. Those rows include 28 positives.
- **Skew in CHL:** the log transform reduces the skewness of the imputed train values from 5.40 to 0.98. Scaled train values now run from -3.14 to 4.27, where the largest was about 9.6 without the log.
- **Applying this at prediction time:** the steps must run in the same order, using `log10(CHL)` with the train mean and standard deviation of the logged values (0.501 and 0.303).
- **Columns left out:** `Season` and `spatial_cluster` are categories, so they still need encoding for models that need numeric inputs. `Date`, `Day`, `Month` and `Year` are unscaled, and the manual describes them as redundant.